# TP : Analyse de l'inflation avec Python

**Données :** inflation trimestrielle (variation annuelle en % de l'indice des prix à la consommation), 2013 T1 à 2026 T2.

## Objectifs
1. identifier le **type de problème** et justifier ce choix ;
2. charger le fichier Excel et l'explorer ;
3. faire des **requêtes** sur les données (pandas) ;
4. produire des **graphiques** et les interpréter ;
5. préparer les données sans fuite de données (découpage chronologique) ;
6. comparer plusieurs modèles à partir d'une référence simple ;
7. évaluer, interpréter et conclure.

## 1. De quel type de problème s'agit-il ?
Nous avons **une seule variable** (l'inflation) mesurée à intervalles réguliers : c'est une **série temporelle**.

- Pour **décrire** l'inflation (hausse, baisse, pic), on fait de l'**analyse descriptive**.
- Pour **prévoir** l'inflation du prochain trimestre à partir de ses valeurs passées, on fait de l'**apprentissage supervisé, plus précisément de la régression** : la cible est un nombre réel (en %), pas une catégorie.
- Différence avec le TP des manchots : ici l'ordre des données compte. On ne mélange donc **jamais** les lignes avant de découper entraînement / test : on garde le passé pour apprendre et le futur pour tester.

## 2. Chargement des données

In [ ]:
# Importation des bibliothèques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproductibilité et style des graphiques
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Chargement du fichier Excel
# Dans Google Colab : une fenêtre s'ouvre, choisis le fichier "Inflation_-_Tableau.xlsx"
try:
    from google.colab import files
    televerse = files.upload()
    FICHIER = list(televerse.keys())[0]
except ImportError:
    FICHIER = "Inflation_-_Tableau.xlsx"   # si tu exécutes en local

# Les données commencent à la ligne 2, colonnes B et C
brut = pd.read_excel(FICHIER, header=1, usecols="B:C")
brut.columns = ["Trimestre", "Inflation"]
brut = brut.dropna(subset=["Trimestre"]).reset_index(drop=True)
print("Dimensions brutes :", brut.shape)
brut.head(8)

## 3. Nettoyage et construction des colonnes
Dans le fichier, l'année n'est écrite que sur la ligne T1 (« 2013 T1 », puis « T2 », « T3 », « T4 »). Nous la **recopions vers le bas** pour avoir une année et un trimestre sur chaque ligne, puis nous créons une vraie **date**.

In [ ]:
df = brut.copy()
df["Trimestre"] = df["Trimestre"].astype(str).str.strip()

# Année : extraite de « 2013 T1 », puis recopiée vers le bas
df["Annee"] = df["Trimestre"].str.extract(r"(\d{4})")[0].ffill().astype(int)
# Trimestre : le chiffre après le T
df["T"] = df["Trimestre"].str.extract(r"T(\d)")[0].astype(int)

# Date = fin du trimestre (pratique pour les graphiques)
df["Date"] = pd.PeriodIndex.from_fields(year=df["Annee"], quarter=df["T"], freq="Q").to_timestamp(how="end").normalize()
df["Periode"] = df["Annee"].astype(str) + " T" + df["T"].astype(str)

# Lignes sans valeur d'inflation (trimestres pas encore publiés)
print("Trimestres sans valeur :", df.loc[df["Inflation"].isna(), "Periode"].tolist())
df = df.dropna(subset=["Inflation"]).reset_index(drop=True)
df = df[["Periode", "Annee", "T", "Date", "Inflation"]]
print("Observations utilisables :", len(df))
df.head()

## 4. Première exploration

In [ ]:
df.info()
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

In [ ]:
# Statistiques descriptives de l'inflation
df["Inflation"].describe().round(2)

**Lecture.** L'inflation moyenne est d'environ 1,9 % et la médiane de 1,25 %. L'écart-type est élevé (2,2) et le maximum (9,1 %) est très loin de la médiane : il y a une période exceptionnelle dans les données. Nous allons la repérer avec des requêtes.

## 5. Requêtes sur les données
Les requêtes servent à poser des questions précises au tableau. On utilise `query`, le filtrage booléen et `groupby`.

In [ ]:
# Q1 : Quel est le trimestre où l'inflation est la plus haute ? la plus basse ?
print("Maximum :", df.loc[df["Inflation"].idxmax(), ["Periode", "Inflation"]].tolist())
print("Minimum :", df.loc[df["Inflation"].idxmin(), ["Periode", "Inflation"]].tolist())

In [ ]:
# Q2 : Quels trimestres ont une inflation supérieure à 4 % ?
df.query("Inflation > 4")[["Periode", "Inflation"]]

In [ ]:
# Q3 : Quels trimestres ont une inflation négative (déflation) ?
df.query("Inflation < 0")[["Periode", "Inflation"]]

In [ ]:
# Q4 : Inflation moyenne, minimale et maximale par année
par_annee = df.groupby("Annee")["Inflation"].agg(["mean", "min", "max"]).round(2)
par_annee

In [ ]:
# Q5 : Quelle année a eu l'inflation moyenne la plus élevée ?
print("Année la plus inflationniste :", par_annee["mean"].idxmax(), "->", par_annee["mean"].max(), "%")
print("Année la moins inflationniste :", par_annee["mean"].idxmin(), "->", par_annee["mean"].min(), "%")

In [ ]:
# Q6 : Combien de trimestres sont au-dessus de 2 % (repère habituel de stabilité des prix) ?
au_dessus = (df["Inflation"] > 2).sum()
print(f"{au_dessus} trimestres sur {len(df)} ({au_dessus/len(df):.0%}) dépassent 2 %")

In [ ]:
# Q7 : Comparer trois périodes : avant 2022, la crise 2022-2023, et après 2023
def periode(a):
    if a < 2022: return "1) Avant 2022"
    if a <= 2023: return "2) Crise 2022-2023"
    return "3) Après 2023"

df["Phase"] = df["Annee"].apply(periode)
df.groupby("Phase")["Inflation"].agg(["count", "mean", "std", "max"]).round(2)

In [ ]:
# Q8 : Variation de l'inflation d'un trimestre à l'autre : quelles sont les plus fortes hausses et baisses ?
df["Variation"] = df["Inflation"].diff()
print("5 plus fortes hausses :")
print(df.nlargest(5, "Variation")[["Periode", "Inflation", "Variation"]].round(2).to_string(index=False))
print("\n5 plus fortes baisses :")
print(df.nsmallest(5, "Variation")[["Periode", "Inflation", "Variation"]].round(2).to_string(index=False))

**Lecture.** Les valeurs extrêmes se concentrent en 2022-2023 : l'inflation passe d'environ 0 % début 2021 à plus de 9 % début 2023, puis redescend. Avant et après cette crise, l'inflation reste faible et peu variable.

## 6. Visualisation des données

### 6.1 Évolution de l'inflation dans le temps

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df["Date"], df["Inflation"], marker="o", markersize=4, linewidth=1.8, color="tab:blue")
ax.axhline(2, color="grey", linestyle="--", label="repère 2 %")
ax.axhline(0, color="black", linewidth=0.8)
ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31"), color="tab:red", alpha=0.12, label="crise 2022-2023")
i = df["Inflation"].idxmax()
ax.annotate(f"pic : {df.loc[i,'Inflation']:.1f} % ({df.loc[i,'Periode']})",
            (df.loc[i, "Date"], df.loc[i, "Inflation"]),
            xytext=(-150, -10), textcoords="offset points", arrowprops=dict(arrowstyle="->"))
ax.set_title("Inflation trimestrielle (variation annuelle de l'IPC)")
ax.set_xlabel("Date")
ax.set_ylabel("Inflation (%)")
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** La courbe est basse et instable jusqu'en 2021, monte très fortement en 2022, atteint son pic début 2023, puis redescend rapidement. Depuis 2024, elle revient autour de 0 à 2 %, avec quelques valeurs négatives.

### 6.2 Inflation moyenne par année

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
couleurs = ["tab:red" if v > 4 else "tab:blue" for v in par_annee["mean"]]
ax.bar(par_annee.index.astype(str), par_annee["mean"], color=couleurs)
for x, v in zip(par_annee.index.astype(str), par_annee["mean"]):
    ax.text(x, v + (0.15 if v >= 0 else -0.5), f"{v:.1f}", ha="center", fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Inflation moyenne par année")
ax.set_xlabel("Année")
ax.set_ylabel("Inflation moyenne (%)")
plt.tight_layout()
plt.show()

**Lecture.** Les années 2022 et 2023 ressortent nettement (barres rouges). Toutes les autres années ont une moyenne inférieure à 2 %.

### 6.3 Distribution et boîtes à moustaches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.histplot(df["Inflation"], bins=12, kde=True, ax=axes[0], color="tab:blue")
axes[0].axvline(df["Inflation"].mean(), color="red", linestyle="--", label="moyenne")
axes[0].axvline(df["Inflation"].median(), color="green", linestyle="--", label="médiane")
axes[0].set_title("Distribution de l'inflation")
axes[0].set_xlabel("Inflation (%)")
axes[0].set_ylabel("Nombre de trimestres")
axes[0].legend()

sns.boxplot(data=df, x="Phase", y="Inflation", hue="Phase", legend=False, ax=axes[1])
axes[1].set_title("Inflation selon la période")
axes[1].set_xlabel("")
axes[1].set_ylabel("Inflation (%)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

**Lecture.** La distribution est **asymétrique à droite** : la moyenne est tirée vers le haut par quelques trimestres de crise, donc la moyenne dépasse la médiane. La boîte « Crise 2022-2023 » est très haute par rapport aux deux autres.

### 6.4 Moyenne mobile et saisonnalité

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Moyenne mobile sur 4 trimestres (= 1 an) : lisse les variations
df["Moy_mobile_4T"] = df["Inflation"].rolling(4).mean()
axes[0].plot(df["Date"], df["Inflation"], alpha=0.5, label="Inflation")
axes[0].plot(df["Date"], df["Moy_mobile_4T"], color="tab:red", linewidth=2.5, label="Moyenne mobile 4 trimestres")
axes[0].set_title("Tendance lissée")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Inflation (%)")
axes[0].legend()

# Y a-t-il un effet trimestre ?
sns.boxplot(data=df, x="T", y="Inflation", hue="T", legend=False, ax=axes[1])
axes[1].set_title("Inflation selon le trimestre (T1 à T4)")
axes[1].set_xlabel("Trimestre")
axes[1].set_ylabel("Inflation (%)")

plt.tight_layout()
plt.show()

**Lecture.** La moyenne mobile fait ressortir la tendance : stable, puis une grosse vague 2022-2023, puis un retour au calme. Les quatre trimestres ont des distributions proches : il n'y a pas de saisonnalité nette, ce qui est normal car la variable est une variation **annuelle** (déjà comparée au même trimestre de l'année précédente).

### 6.5 Carte de chaleur année × trimestre

In [ ]:
tableau = df.pivot(index="Annee", columns="T", values="Inflation")
fig, ax = plt.subplots(figsize=(6.5, 6.5))
sns.heatmap(tableau, annot=True, fmt=".1f", cmap="coolwarm", center=2,
            cbar_kws={"label": "Inflation (%)"}, ax=ax)
ax.set_title("Inflation par année et par trimestre")
ax.set_xlabel("Trimestre")
ax.set_ylabel("Année")
plt.tight_layout()
plt.show()

**Lecture.** Les cases rouges (2022 T2 à 2023 T3) montrent la crise d'un coup d'œil. Les cases bleues, proches de zéro ou négatives, apparaissent surtout en 2019 et depuis fin 2025.

### 6.6 Le passé aide-t-il à prédire le futur ? (autocorrélation)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x=df["Inflation"].shift(1), y=df["Inflation"], ax=ax, s=60)
lim = [df["Inflation"].min() - 0.5, df["Inflation"].max() + 0.5]
ax.plot(lim, lim, "--", color="grey", label="y = x")
ax.set_xlabel("Inflation du trimestre précédent (%)")
ax.set_ylabel("Inflation du trimestre (%)")
ax.set_title("Inflation en fonction de la valeur précédente")
ax.legend()
plt.tight_layout()
plt.show()

print("Corrélation avec le trimestre précédent :", round(df["Inflation"].corr(df["Inflation"].shift(1)), 2))

**Lecture.** Les points suivent la diagonale : l'inflation d'un trimestre ressemble à celle du trimestre précédent. C'est ce qui rend la **prévision** possible, et c'est pourquoi on utilisera les valeurs passées (« retards ») comme variables d'entrée.

## 7. Préparation des données pour la prévision
Règles à respecter :
- **Pas de mélange aléatoire** : on découpe dans l'ordre du temps (le passé entraîne, le futur teste).
- **Variables d'entrée** : les inflations des 4 trimestres précédents (retards 1 à 4).
- **Cible** : l'inflation du trimestre.
- Les premières lignes perdent des valeurs (pas de passé pour 2013) : on les retire.

In [ ]:
data = df[["Periode", "Inflation"]].copy()
for k in range(1, 5):
    data[f"retard_{k}"] = data["Inflation"].shift(k)
data = data.dropna().reset_index(drop=True)

variables = ["retard_1", "retard_2", "retard_3", "retard_4"]
X, y = data[variables], data["Inflation"]

# Découpage chronologique : les 10 derniers trimestres servent de test
n_test = 10
X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]

print("Entraînement :", data["Periode"].iloc[0], "->", data["Periode"].iloc[len(X_train)-1], f"({len(X_train)} trimestres)")
print("Test         :", data["Periode"].iloc[len(X_train)], "->", data["Periode"].iloc[-1], f"({len(X_test)} trimestres)")

## 8. Entraînement et comparaison de modèles
### 8.1 Principe
Nous comparons des modèles du plus simple au plus complexe. Deux **références** :
- **Moyenne** : prédit toujours la moyenne de l'entraînement ;
- **Persistance** : prédit que l'inflation sera la même qu'au trimestre précédent (très difficile à battre en économie).

La comparaison utilise une **validation croisée temporelle** (`TimeSeriesSplit`) : chaque pli teste toujours après la période d'apprentissage. Métrique : **MAE** (erreur absolue moyenne, en points de pourcentage). Plus elle est petite, mieux c'est.

In [ ]:
modeles = {
    "Référence : moyenne": DummyRegressor(strategy="mean"),
    "Régression linéaire": LinearRegression(),
    "Ridge (régularisée)": Ridge(alpha=1.0),
    "Forêt aléatoire": RandomForestRegressor(n_estimators=200, random_state=42),
}

validation = TimeSeriesSplit(n_splits=5)
resultats = []

# Référence persistance : on prédit la valeur du trimestre précédent
mae_persistance = mean_absolute_error(y_train, X_train["retard_1"])
resultats.append({"Modèle": "Référence : persistance", "MAE moyenne": mae_persistance, "Écart-type": 0.0})

for nom, m in modeles.items():
    scores = -cross_val_score(m, X_train, y_train, cv=validation, scoring="neg_mean_absolute_error")
    resultats.append({"Modèle": nom, "MAE moyenne": scores.mean(), "Écart-type": scores.std()})

tableau_modeles = pd.DataFrame(resultats).sort_values("MAE moyenne").reset_index(drop=True)
tableau_modeles.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(tableau_modeles["Modèle"], tableau_modeles["MAE moyenne"],
        xerr=tableau_modeles["Écart-type"], capsize=5,
        color=sns.color_palette("Set2", len(tableau_modeles)))
ax.invert_yaxis()
for i, v in enumerate(tableau_modeles["MAE moyenne"]):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center")
ax.set_xlabel("Erreur absolue moyenne (points de %, validation croisée temporelle)")
ax.set_title("Comparaison des modèles : plus petit = meilleur")
plt.tight_layout()
plt.show()

**Lecture.** Un modèle n'est utile que s'il fait **mieux que les références**. Ici, la **persistance** (répéter la valeur précédente) fait mieux que tous les modèles appris en validation croisée, et les modèles linéaires font bien mieux que la moyenne. La forêt aléatoire est la moins bonne des modèles appris : avec seulement une quarantaine d'observations d'entraînement, un modèle complexe n'a pas assez de données. Les barres d'erreur (écart-type) sont grandes : il faut rester prudent dans les conclusions.

## 9. Évaluation finale sur le jeu de test
Le meilleur modèle (hors références) est entraîné sur tout l'entraînement, puis testé **une seule fois** sur les 10 derniers trimestres.

In [ ]:
candidats = tableau_modeles[~tableau_modeles["Modèle"].str.startswith("Référence")]
meilleur_nom = candidats.iloc[0]["Modèle"]
meilleur = modeles[meilleur_nom].fit(X_train, y_train)
pred = meilleur.predict(X_test)
pred_persist = X_test["retard_1"].values

def metriques(vrai, prevu):
    return {"MAE": mean_absolute_error(vrai, prevu),
            "RMSE": np.sqrt(mean_squared_error(vrai, prevu)),
            "R²": r2_score(vrai, prevu)}

bilan = pd.DataFrame({meilleur_nom: metriques(y_test, pred),
                      "Référence : persistance": metriques(y_test, pred_persist)}).T
print("Meilleur modèle :", meilleur_nom)
bilan.round(3)

### 9.1 Prévisions contre réalité

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
periodes_test = data["Periode"].iloc[-n_test:]
ax.plot(periodes_test, y_test.values, "o-", linewidth=2, label="Réel")
ax.plot(periodes_test, pred, "s--", linewidth=2, label=f"Prévu ({meilleur_nom})")
ax.plot(periodes_test, pred_persist, "^:", color="grey", label="Persistance")
ax.set_title("Inflation réelle et prévue sur la période de test")
ax.set_xlabel("Trimestre")
ax.set_ylabel("Inflation (%)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Sur le test, le modèle fait à peine mieux que la persistance (voir la MAE). Le **R² est négatif** : le modèle prédit moins bien que la simple moyenne de la période de test. Cela s'explique par le fait que l'inflation varie peu sur 2024-2026, alors que le modèle a appris sur une période incluant la crise de 2022-2023. Les prévisions suivent la tendance mais avec un décalage, car elles s'appuient sur le passé : un modèle ne peut pas deviner un choc nouveau. C'est la limite principale de la prévision de l'inflation.

### 9.2 Erreurs de prévision (résidus)

In [ ]:
residus = y_test.values - pred
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(periodes_test, residus, color=["tab:red" if r > 0 else "tab:blue" for r in residus])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Erreur = réel - prévu (rouge : sous-estimation, bleu : surestimation)")
ax.set_xlabel("Trimestre")
ax.set_ylabel("Erreur (points de %)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Lecture.** Les erreurs positives (rouges) sont des trimestres où l'inflation réelle a été plus haute que prévu, les négatives (bleues) l'inverse. Une erreur qui garde toujours le même signe indiquerait un modèle biaisé.

## 10. Interprétation du modèle

In [ ]:
# Poids des retards (régression linéaire entraînée sur tout l'entraînement)
lin = LinearRegression().fit(X_train, y_train)
poids = pd.Series(lin.coef_, index=variables)

fig, ax = plt.subplots(figsize=(7, 4))
poids.plot.bar(ax=ax, color="teal")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Poids de chaque retard dans la régression linéaire")
ax.set_ylabel("Coefficient")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print(poids.round(3))

**Lecture.** Le retard 1 (trimestre précédent) a de loin le poids le plus important : c'est la valeur la plus récente qui informe le mieux sur la valeur suivante. Les autres retards ont des poids faibles et négatifs : ils corrigent un peu le retard 1. Attention : ces retards sont corrélés entre eux, les coefficients individuels sont donc à interpréter avec prudence. Cela décrit une **relation statistique**, pas une cause.

## 11. Prévoir le prochain trimestre
Le fichier contient des lignes vides pour les trimestres à venir. Utilisons le modèle pour proposer une valeur pour le **premier trimestre manquant**.

In [ ]:
dernier = df["Inflation"].values[::-1][:4]   # 4 dernières valeurs, de la plus récente à la plus ancienne
nouveau = pd.DataFrame([dict(zip(variables, dernier))])

modele_final = modeles[meilleur_nom].fit(X, y)   # réentraîné sur toutes les données
prevision = modele_final.predict(nouveau)[0]
print("Dernières valeurs utilisées (retard 1 à 4) :", np.round(dernier, 2))
print(f"Prévision pour le prochain trimestre : {prevision:.2f} %")

**Lecture.** Cette valeur est une **estimation indicative**, pas une certitude : l'erreur typique du modèle (voir la MAE de la section 9) donne l'ordre de grandeur de l'incertitude.

## 12. Synthèse et questions

### Ce qu'il faut retenir
1. **Type de problème** : série temporelle ; la prévision relève de la régression supervisée.
2. **Démarche** : charger, nettoyer, explorer par requêtes et graphiques, découper dans l'ordre du temps, comparer à des références, valider, évaluer une seule fois sur le test.
3. **Résultat principal** : l'inflation a été faible et stable, sauf une crise forte en 2022-2023 (pic à 9,1 % en 2023 T1), suivie d'un retour rapide à un niveau bas.
4. **Limite** : un modèle basé sur le passé réagit avec retard aux chocs ; il est difficile de battre la simple persistance.

### Questions
1. Pourquoi ne faut-il pas mélanger les lignes avant de découper entraînement / test dans une série temporelle ?
2. Pourquoi la référence « persistance » est-elle plus exigeante que la référence « moyenne » ?
3. Changez le nombre de retards (2, 6, 8). La MAE s'améliore-t-elle ?
4. Testez avec `n_test = 6` puis `n_test = 14`. Les conclusions changent-elles ?
5. Quelles autres variables économiques (prix du pétrole, taux de change, taux d'intérêt) pourraient améliorer la prévision ?